# Seleção de participantes - Escolaridade (high x medium x low)

Subamostra pareada por escolaridade usada nas análises `Education/Filtered` (Gordon e Glasser). Como `low` é o grupo menor (87 no pool de 881), formam-se trios high-low-medium.

**Regra de pareamento**

* Âncoras: participantes `high`, percorridos na ordem da planilha.
* Cada trio tem os mesmos `sex`, `handedness` e `religious_now`.
* Para cada high, escolhe-se o par (low, medium) ainda disponível que minimiza `d(high,low) + d(high,medium) + d(low,medium)`, com `d = |Δage| + |Δbackground_SES|`. Em empates vale o primeiro par encontrado (medium no laço externo, low no interno).
* Quando acabam os low de um estrato, os demais high desse estrato ficam de fora.
* `pair_id` identifica o trio. `original_index` é a posição do low/medium na lista daquele nível (vazio para o high).
* Resultado: 85 trios = 255 participantes. Dois low (mulheres canhotas religiosas) ficam de fora por não haver high no estrato.

Na análise do paper (high x low), o metadata remove os `medium` (ver `filter_metadata_remove_medium.py`).

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats

# Planilha com os 928 participantes do ID1000 (participants.tsv do OpenNeuro ds003097 + colunas usadas no projeto)
PARTICIPANTS_CSV = "https://docs.google.com/spreadsheets/d/13n8f5tILuIVF6Cxi2izX020_fYNjNNZk/export?format=csv"

# Pasta local de dados do projeto (a que contém ID1000Files/). Rode o notebook a partir dela ou ajuste aqui.
DATA_ROOT = "."
OUTPUT_DIR = DATA_ROOT

OUTPUT_XLSX = os.path.join(OUTPUT_DIR, "selected_participantsEducation.xlsx")

df = pd.read_csv(PARTICIPANTS_CSV)
df.shape

(928, 31)

Identificar sujeitos que não têm funcional

In [2]:
# Remove sujeitos que têm anat/ mas não têm func/ (sem fMRI funcional): 928 -> 881
base_dir = os.path.join(DATA_ROOT, "ID1000Files", "raw")

anat_only = []
for dataset in os.listdir(base_dir):
    dataset_dir = os.path.join(base_dir, dataset)
    if not os.path.isdir(dataset_dir):
        continue
    subs = [d for d in os.listdir(dataset_dir) if d.startswith("sub-") and os.path.isdir(os.path.join(dataset_dir, d))]
    for subj in subs:
        anat_dir = os.path.join(dataset_dir, subj, "anat")
        func_dir = os.path.join(dataset_dir, subj, "func")
        has_anat = os.path.isdir(anat_dir) and any(os.path.isfile(os.path.join(anat_dir, f)) for f in os.listdir(anat_dir))
        has_func = os.path.isdir(func_dir) and any(os.path.isfile(os.path.join(func_dir, f)) for f in os.listdir(func_dir))
        if has_anat and not has_func:
            anat_only.append(subj)

df = df[~df["participant_id"].isin(anat_only)]
print("sujeitos sem func:", len(anat_only), "| pool:", len(df))

sujeitos sem func: 47 | pool: 881


## Pareamento em trios

In [3]:
# Pareamento em trios high-low-medium
EXATAS = ["sex", "handedness", "religious_now"]   # variáveis pareadas exatamente

high = df[df["education_level"] == "high"]                         # âncoras, na ordem da planilha
low = df[df["education_level"] == "low"].reset_index(drop=True)    # original_index = posição aqui
medium = df[df["education_level"] == "medium"].reset_index(drop=True)

def dist(a, b):
    return abs(a["age"] - b["age"]) + abs(a["background_SES"] - b["background_SES"])

usados_low, usados_med = set(), set()
linhas, pair_id = [], 0
for _, h in high.iterrows():
    cand_low = low[~low.index.isin(usados_low)]
    cand_med = medium[~medium.index.isin(usados_med)]
    for col in EXATAS:
        cand_low = cand_low[cand_low[col] == h[col]]
        cand_med = cand_med[cand_med[col] == h[col]]
    if cand_low.empty or cand_med.empty:
        continue   # acabaram os low (ou medium) deste estrato

    # Escolhe o par (medium, low) que minimiza a soma das distâncias entre os três.
    # Empates: vale o primeiro encontrado (medium no laço externo, low no interno).
    melhor, escolha = None, None
    for mi, m in cand_med.iterrows():
        for li, l in cand_low.iterrows():
            custo = dist(h, l) + dist(h, m) + dist(l, m)
            if melhor is None or custo < melhor:
                melhor, escolha = custo, (li, mi)

    li, mi = escolha
    usados_low.add(li)
    usados_med.add(mi)
    for r, oi in [(h, np.nan), (low.loc[li], li), (medium.loc[mi], mi)]:
        r = r.copy()
        r["pair_id"] = pair_id
        r["original_index"] = oi
        linhas.append(r)
    pair_id += 1

df_final = pd.DataFrame(linhas).reset_index(drop=True)
print(df_final["education_level"].value_counts())
df_final

education_level
high      85
low       85
medium    85
Name: count, dtype: int64


,participant_id,age,sex,handedness,BMI,education_level,background_SES,IST_fluid,IST_memory,IST_crystallised,...,gender_identity_M,gender_identity_F,religious_upbringing,religious_now,religious_importance,DWI_TR_run1,DWI_TR_run2,DWI_TR_run3,pair_id,original_index
0,sub-0031,24.75,female,right,24,high,4.5,97.0,64.0,45.0,...,NaN,NaN,yes,yes,1.0,6.311,6.311,6.311,0,NaN
1,sub-0898,23.25,female,right,33,low,4.5,78.0,49.0,29.0,...,NaN,NaN,no,yes,5.0,6.312,6.312,6.312,0,21.0
2,sub-0429,24.25,female,right,20,medium,4.5,90.0,48.0,29.0,...,1.0,7.0,yes,yes,1.0,6.312,6.312,6.312,0,7.0
3,sub-0069,23.50,female,right,23,high,2.0,81.0,42.0,27.0,...,NaN,NaN,yes,yes,1.0,6.312,6.312,6.312,1,NaN
4,sub-0364,24.25,female,right,23,low,2.0,101.0,45.0,30.0,...,NaN,NaN,yes,yes,1.0,6.312,6.312,6.312,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250,sub-0461,22.00,male,right,29,low,2.0,59.0,48.0,23.0,...,7.0,1.0,no,no,NaN,6.311,6.311,6.311,83,58.0
251,sub-0175,22.00,male,right,32,medium,2.5,88.0,58.0,37.0,...,NaN,NaN,yes,no,NaN,6.313,6.313,6.313,83,152.0
252,sub-0196,21.75,male,right,24,high,4.5,100.0,57.0,29.0,...,NaN,NaN,no,no,NaN,6.313,6.313,6.313,84,NaN
253,sub-0632,25.75,male,right,42,low,2.0,127.0,60.0,49.0,...,7.0,1.0,no,no,NaN,6.311,6.311,6.311,84,74.0


## Verificação do balanço

In [4]:
# Balanço entre os três níveis
for var in ["age", "background_SES", "IST_intelligence_total"]:
    grupos = [g[var].dropna() for _, g in df_final.groupby("education_level")]
    f, p = stats.f_oneway(*grupos)
    medias = df_final.groupby("education_level")[var].mean().round(2).to_dict()
    print(f"{var:24s} {medias}  ANOVA F={f:.3f} p={p:.3f}")

# high x low (comparação reportada no paper)
hl = df_final[df_final["education_level"].isin(["high", "low"])]
t, p = stats.ttest_ind(hl.loc[hl.education_level == "high", "age"], hl.loc[hl.education_level == "low", "age"])
print(f"idade high x low: t={t:.3f} p={p:.3f}")

# Variáveis exatas constantes dentro de cada trio
print("exatas constantes nos trios:", all((df_final.groupby("pair_id")[c].nunique() == 1).all() for c in EXATAS))

age                      {'high': 23.02, 'low': 23.11, 'medium': 23.05}  ANOVA F=0.073 p=0.930
background_SES           {'high': 4.52, 'low': 3.32, 'medium': 3.91}  ANOVA F=26.048 p=0.000
IST_intelligence_total   {'high': 217.88, 'low': 171.01, 'medium': 189.98}  ANOVA F=33.030 p=0.000
idade high x low: t=-0.368 p=0.713
exatas constantes nos trios: True


## Gravação

In [6]:
df_final.to_excel(OUTPUT_XLSX, index=False)